# PA4
Group PA4 9, Melker Gustafsson, Pontus Gideflod, Ismail Sacic

## Task 1

In [1]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Perceptron
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

X1 = [{'city':'Gothenburg', 'month':'July'},
      {'city':'Gothenburg', 'month':'December'},
      {'city':'Paris', 'month':'July'},
      {'city':'Paris', 'month':'December'}]
Y1 = ['rain', 'rain', 'sun', 'rain']

X2 = [{'city':'Sydney', 'month':'July'},
      {'city':'Sydney', 'month':'December'},
      {'city':'Paris', 'month':'July'},
      {'city':'Paris', 'month':'December'}]
Y2 = ['rain', 'sun', 'sun', 'rain']

classifier1 = make_pipeline(DictVectorizer(), Perceptron(max_iter=10))
classifier1.fit(X1, Y1)
guesses1 = classifier1.predict(X1)
print(accuracy_score(Y1, guesses1))

classifier2 = make_pipeline(DictVectorizer(), Perceptron(max_iter=10))
#classifier2 = make_pipeline(DictVectorizer(), LinearSVC())
classifier2.fit(X2, Y2)
guesses2 = classifier2.predict(X2)
print(accuracy_score(Y2, guesses2))

1.0
0.5


### When we use the second set instead of the first, for some strange reason our classifier performs poorly! We can't improve it by switching to a LinearSVC. Do you have an idea what's going on? Why could the classifier "memorize" the training data in the first case, but not in the second case?

The DictVectorizer turns the dataset into points with either the result RAIN or SUN. the first dataset then becomes (0,1) = RAIN, (1,0) = SUN, (0,0) = RAIN and (1,1) = RAIN and if we plot this on a graph we can quickly see that a linear line can easily seperate the dataset into RAIN on one side and SUN on the other. On the second dataset however, we end up with (0,1) = SUN, (1,0) = SUN, (0,0) = RAIN and (1,1) = RAIN which means that the RAIN and SUN results are no longer seperable linearly (by a straight line). This causes the classifier to fail and in order to improve this you would need to change to a nonlinear model.

## Task 2

### The Perceptron package to install contains a Python program (doc_classification.py) that carries out an experiment in document classification. This is the same file that we used in one of the demonstrations for one of the lectures. The task here is to determine whether a product review is positive or negative. The program trains a classifier using our perceptron implementation, and then evaluates the classifier on a test set. Run the experiment code and make sure it works on your machine. Training should take at most a few seconds, and the accuracy should be about 0.80.

We had to make a small alteration to add a is_trained flag to the Perceptron class so that Sklearn could understand that the model was fit already when running predict. After that 
we ran the code and got Accuracy: 0.7919, Training time: 0.90 sec.

## Task 3

### Implement the Pegasos algorithm for training support vector classifiers by converting the pseudocode in Algorithm 1 in the clarification document (Figure 1 in the original Pegasos paper) into proper Python. Test your implementation by using your own classifier instead of the perceptron in doc_clasification.py. It's probably easiest if you start from the existing code, for instance by making a copy of the class Perceptron, and then just modify it to implement Algorithm 1.

In [2]:
import numpy as np
from sklearn.base import BaseEstimator
from sklearn.utils.validation import check_is_fitted
class LinearClassifier(BaseEstimator):
    """
    General class for binary linear classifiers. Implements the predict
    function, which is the same for all binary linear classifiers. There are
    also two utility functions.
    """

    def decision_function(self, X):
        """
        Computes the decision function for the inputs X. The inputs are assumed to be
        stored in a matrix, where each row contains the features for one
        instance.
        """
        return X.dot(self.w)

    def predict(self, X):
        """
        Predicts the outputs for the inputs X. The inputs are assumed to be
        stored in a matrix, where each row contains the features for one
        instance.
        """
        check_is_fitted(self, ['is_fitted_']) 
        
        scores = self.decision_function(X)

        out = np.select([scores >= 0.0, scores < 0.0],
                        [self.positive_class, self.negative_class],
                        default=self.negative_class) 
        return out

    def find_classes(self, Y):
        """
        Finds the set of output classes in the output part Y of the training set.
        If there are exactly two classes, one of them is associated to positive
        classifier scores, the other one to negative scores. If the number of
        classes is not 2, an error is raised.
        """
        classes = sorted(set(Y))
        if len(classes) != 2:
            raise Exception("this does not seem to be a 2-class problem")
        self.positive_class = classes[1]
        self.negative_class = classes[0]
        self.is_fitted_ = True

    def encode_outputs(self, Y):
        """
        A helper function that converts all outputs to +1 or -1.
        """
        return np.array([1 if y == self.positive_class else -1 for y in Y])

class Pegasos(LinearClassifier):
    """
    A straightforward implementation of the perceptron learning algorithm.
    """

    def __init__(self, n_iter=20):
        """
        The constructor can optionally take a parameter n_iter specifying how
        many times we want to iterate through the training set.
        """
        self.n_iter = n_iter

    def fit(self, X, Y, lambda_=0.0):
        """
        Train a linear classifier using the perceptron learning algorithm.
        """

        # First determine which output class will be associated with positive
        # and negative scores, respectively.
        self.find_classes(Y)

        # Convert all outputs to +1 (for the positive class) or -1 (negative).
        Ye = self.encode_outputs(Y)

        # If necessary, convert the sparse matrix returned by a vectorizer
        # into a normal NumPy matrix.
        if not isinstance(X, np.ndarray):
            X = X.toarray()

        # Initialize the weight vector to all zeros.
        n_features, n_samples = X.shape[1], X.shape[0]
        self.w = np.zeros(n_features)
        
        t = 0
        # Pegasos algorithm:
        for epoch in range(self.n_iter):
            indices = np.random.permutation(n_samples)
            for i in indices:
                t += 1
                eta = 1.0 / (lambda_ * t)
                
                # Compute the output score for this instance.
                x_i, y_i = X[i], Ye[i]
                score = x_i.dot(self.w)
                # If there was an error, update the weights.
                if y_i*score < 1:
                    self.w = (1 - eta * lambda_) * self.w + eta * y_i * x_i
                else:
                    self.w = (1 - eta * lambda_) * self.w

        return self


Above is our implementation of the Pegasos algorithm. We copied the Perceptron implementation given and altered the fit method to implement the Pegasos algorithm instead. 

In [3]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from Task3 import Pegasos

def read_data(corpus_file):
    X = []
    Y = []
    with open(corpus_file, encoding='utf-8') as f:
        for line in f:
            _, y, _, x = line.split(maxsplit=3)
            X.append(x.strip())
            Y.append(y)
    return X, Y


if __name__ == '__main__':
    X, Y = read_data('data/all_sentiment_shuffled.txt')

    Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2,
                                                    random_state=0)
    N = 50
    lam = 1 / (N * N)

    svc_pipeline = make_pipeline(
        TfidfVectorizer(),
        SelectKBest(k=1000),
        Normalizer(),
        Pegasos(n_iter=N),
    )

    t0 = time.time()
    svc_pipeline.fit(Xtrain, Ytrain, pegasos__lambda_=lam)
    t1 = time.time()
    Yguess_svc = svc_pipeline.predict(Xtest)
    print('--- Task 3: Pegasos ---')
    print('Training time: {:.2f} sec.'.format(t1 - t0))
    print('Accuracy:      {:.4f}'.format(accuracy_score(Ytest, Yguess_svc)))

--- Task 3: Pegasos ---
Training time: 2.97 sec.
Accuracy:      0.8338


We used the above code to run and test our Pegasos implementation. The code is a slightly altered version of doc_classification. We tested out different parameter values for n_iter and lambda_ and the best accuracy we found was using n_iter = 50 and lambda_ = 1 / (n_iter * n_iter) which gave us an Accuracy of 0.8338 within 2.97 sec Training time. Higher values for n_iter seemed to reduce accuracy and lower did the same. Higher values for lambda reduced accuracy and while lower values of lambda could improve accuracy it could come at the risk of overfitting.

## Task 4